# MusicBrainz で曲メタデータを補完

このノートブックでは、`data_collection` の YouTube Music から取得した CSV を読み込み、MusicBrainz を使って `composer` / `原作者` などのメタデータを補完します。

In [21]:
import csv
import json
import os
import re
import time
from pathlib import Path

import musicbrainzngs
import pandas as pd

## MusicBrainz の設定

MusicBrainz API はユーザーエージェントを必要とします。質問や連絡先を設定してください。

In [22]:
musicbrainzngs.set_useragent(
    'music_tda_tobas_sanjuanito',
    '0.1',
    'oookayamaswallow@gmail.com'
 )

CACHE_DIR = Path('musicbrainz_cache')
CACHE_DIR.mkdir(exist_ok=True, parents=True)

def load_json_cache(filename):
    path = CACHE_DIR / filename
    if path.exists():
        with path.open('r', encoding='utf-8') as f:
            return json.load(f)
    return None

def save_json_cache(filename, data):
    path = CACHE_DIR / filename
    with path.open('w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

## CSV 読み込み

YouTube Music から取得した2つの CSV を読み込み、MusicBrainz で補完します。

In [23]:
csv_paths = [
    Path('./sanjuanito_20260418_2309.csv'),
    Path('./tobas_20260418_2311.csv'),
 ]

def load_source_csv(path):
    return pd.read_csv(path, dtype=str).fillna('')

source_dfs = [load_source_csv(path) for path in csv_paths]
source_dfs[0].head()

,title,artist,album,url,description,release_year,fetch_date
0,Ñucallajta,Proyección,Ámame (Folklore Andino),https://www.youtube.com/watch?v=SeMeKsRY7WU,,,2026-04-18 23:06:41
1,Ecuador San Juanito,Ecuador Manta,Alegrando El Alma,https://www.youtube.com/watch?v=IXVUwOqt5XE,,,2026-04-18 23:06:41
2,Naupa Llacta,Ecuador Causary,Music From The Andes,https://www.youtube.com/watch?v=Y75gLE0ihyA,,,2026-04-18 23:06:41
3,Ñuca Llacta,Ñanda Mañachi,Folklore Ecuatorianisimo,https://www.youtube.com/watch?v=xRei8rigCOA,,,2026-04-18 23:06:41
4,Selección de San Juanitos,Proyección,Toda... Una Vida (La Nueva Expresion del Folkl...,https://www.youtube.com/watch?v=9nuMFtP71jY,,,2026-04-18 23:06:41


## MusicBrainz 検索関数

`title` と `artist` を使って録音を検索し、候補上位を返します。

In [24]:
def normalize_query_text(text):
    if not isinstance(text, str):
        return ''
    text = text.strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def search_recording(title, artist, limit=5):
    title = normalize_query_text(title)
    artist = normalize_query_text(artist)
    if not title:
        return []

    cache_key = f"recording_search_{re.sub(r'[^0-9a-zA-Z_-]', '_', title)}_{re.sub(r'[^0-9a-zA-Z_-]', '_', artist)}.json"
    cached = load_json_cache(cache_key)
    if cached is not None:
        return cached

    try:
        result = musicbrainzngs.search_recordings(recording=title, artist=artist, limit=limit)
        recordings = result.get('recording-list', [])
    except Exception as exc:
        print(f'Error searching recording: {title} / {artist} -> {exc}')
        recordings = []

    save_json_cache(cache_key, recordings)
    time.sleep(1.0)
    return recordings

## 原作者（作曲者）を取得する関数

録音から関連する Work をたどり、Work の作曲者情報を抽出します。

In [25]:
def get_work_composers(work_id):
    if not work_id:
        return []
    cache_key = f'work_{work_id}.json'
    cached = load_json_cache(cache_key)
    if cached is not None:
        work = cached
    else:
        try:
            work = musicbrainzngs.get_work_by_id(work_id, includes=['artist-rels'])
        except Exception as exc:
            print(f'Error getting work {work_id}: {exc}')
            return []
        save_json_cache(cache_key, work)
        time.sleep(1.0)

    work_data = work.get('work', {})
    rels = work_data.get('artist-relation-list', []) or []
    if isinstance(rels, dict):
        rels = [rels]

    composers = []
    for rel in rels:
        if rel.get('type', '').lower() == 'composer':
            artist = rel.get('artist', {}).get('name') or rel.get('artist-credit-name')
            if artist:
                composers.append(artist)
    return composers

def get_artist_country(artist_id):
    if not artist_id:
        return ''
    cache_key = f'artist_{artist_id}.json'
    cached = load_json_cache(cache_key)
    if cached is not None:
        artist = cached
    else:
        try:
            artist = musicbrainzngs.get_artist_by_id(artist_id, includes=['artist-rels'])
        except Exception as exc:
            print(f'Error getting artist {artist_id}: {exc}')
            return ''
        save_json_cache(cache_key, artist)
        time.sleep(1.0)

    artist_data = artist.get('artist', {})
    return artist_data.get('country', '')

def get_recording_metadata(recording):
    record_id = recording.get('id')
    if not record_id:
        return {}

    cache_key = f'recording_{record_id}.json'
    cached = load_json_cache(cache_key)
    if cached is not None:
        rec = cached
    else:
        try:
            rec = musicbrainzngs.get_recording_by_id(record_id, includes=['work-rels', 'artist-rels', 'releases'])
        except Exception as exc:
            print(f'Error getting recording {record_id}: {exc}')
            return {}
        save_json_cache(cache_key, rec)
        time.sleep(1.0)

    recording_data = rec.get('recording', {})
    works = recording_data.get('work-relation-list', []) or []
    if isinstance(works, dict):
        works = [works]

    work_ids = [w.get('work', {}).get('id') for w in works if w.get('work', {}).get('id')]
    composers = []
    for work_id in work_ids:
        composers.extend(get_work_composers(work_id))

    return {
        'recording_id': record_id,
        'work_ids': work_ids,
        'composer_list': sorted(set(composers)),
    }

## MusicBrainz とのマッチング実行

存在する CSV それぞれについて、上位候補を MusicBrainz で検索し、原作者候補を付与します。

In [26]:
def get_artist_names_from_credit(artist_credit):
    if isinstance(artist_credit, list):
        return [ac.get('name', ac.get('artist', {}).get('name', '')) for ac in artist_credit if isinstance(ac, dict)]
    elif isinstance(artist_credit, dict):
        return [artist_credit.get('name', artist_credit.get('artist', {}).get('name', ''))]
    else:
        # If it's a string or other, return as is
        return [str(artist_credit)]

def enrich_dataframe(df):
    enriched_rows = []
    total = len(df)
    for idx, row in df.iterrows():
        title = row.get('title', '')
        artist = row.get('artist', '')

        candidates = search_recording(title, artist, limit=3)
        best = candidates[0] if candidates else None

        metadata = get_recording_metadata(best) if best else {}

        artist_credit = best.get('artist-credit', []) if best else []
        matched_artist = ', '.join(get_artist_names_from_credit(artist_credit))

        # Get artist ID and country
        artist_id = None
        if artist_credit and isinstance(artist_credit, list) and artist_credit:
            first_artist = artist_credit[0].get('artist', {})
            artist_id = first_artist.get('id')
        country = get_artist_country(artist_id) if artist_id else ''

        enriched_rows.append({
            **row.to_dict(),
            'mb_recording_id': metadata.get('recording_id', ''),
            'mb_work_ids': ','.join(metadata.get('work_ids', [])),
            'composer': '; '.join(metadata.get('composer_list', [])),
            'matched_title': best.get('title', '') if best else '',
            'matched_artist': matched_artist,
            'match_score': best.get('score', '') if best else '',
            'artist_country': country,
        })

        if (idx + 1) % 10 == 0 or idx + 1 == total:
            print(f'  processed {idx + 1}/{total} rows')
    return pd.DataFrame(enriched_rows)

for path, df in zip(csv_paths, source_dfs):
    print('Enriching', path.name)
    enriched = enrich_dataframe(df)
    output_path = path.with_name(path.stem + '_musicbrainz_enriched.csv')
    enriched.to_csv(output_path, index=False, encoding='utf-8-sig')
    print('Saved', output_path)

Enriching sanjuanito_20260418_2309.csv
  processed 10/200 rows
  processed 20/200 rows
  processed 30/200 rows
  processed 40/200 rows
  processed 50/200 rows
  processed 60/200 rows
  processed 70/200 rows
  processed 80/200 rows
  processed 90/200 rows
  processed 100/200 rows
  processed 110/200 rows
  processed 120/200 rows
  processed 130/200 rows
  processed 140/200 rows
  processed 150/200 rows
  processed 160/200 rows
  processed 170/200 rows
  processed 180/200 rows
  processed 190/200 rows
  processed 200/200 rows
Saved sanjuanito_20260418_2309_musicbrainz_enriched.csv
Enriching tobas_20260418_2311.csv
  processed 10/200 rows
  processed 20/200 rows
  processed 30/200 rows
  processed 40/200 rows
  processed 50/200 rows
  processed 60/200 rows
  processed 70/200 rows
  processed 80/200 rows
  processed 90/200 rows
  processed 100/200 rows
  processed 110/200 rows
  processed 120/200 rows
  processed 130/200 rows
  processed 140/200 rows
  processed 150/200 rows
  processed 16

## 次のステップ

- `composer` が空の場合は手動確認や追加クエリ
- `spotipy` を使って候補補完をしたい場合は、次に `track` 検索を追加できます。
- キャッシュを活用すると大量曲数でも MusicBrainz のレート制限内で回せます。

## フィルタリング処理

_enrichedファイルを読み込み、指定された条件でフィルタリングして新しいファイルに保存します。

In [27]:
enriched_files = [
    Path('./sanjuanito_20260418_2309_musicbrainz_enriched.csv'),
    Path('./tobas_20260418_2311_musicbrainz_enriched.csv'),
]

for path in enriched_files:
    print(f'Processing {path.name}')
    df = pd.read_csv(path, dtype=str).fillna('')
    
    # Step 1: Country filter
    if 'sanjuanito' in path.name:
        df_filtered = df[df['artist_country'].isin(['EC', ''])]
    else:  # tobas
        df_filtered = df[df['artist_country'].isin(['BO', ''])]
    
    output_path1 = path.with_name(path.stem.replace('_musicbrainz_enriched', '_filtered_country') + '.csv')
    df_filtered.to_csv(output_path1, index=False, encoding='utf-8-sig')
    print(f'Saved {output_path1.name} ({len(df_filtered)} rows)')
    
    # Step 2: Remove remix (selection / Selección)
    df_filtered2 = df_filtered[~df_filtered['title'].str.contains('selection|Selección', case=False, na=False)]
    
    output_path2 = path.with_name(path.stem.replace('_musicbrainz_enriched', '_filtered_country_remix') + '.csv')
    df_filtered2.to_csv(output_path2, index=False, encoding='utf-8-sig')
    print(f'Saved {output_path2.name} ({len(df_filtered2)} rows)')
    
    # Step 3: Artist match (artist in matched_artist, case insensitive)
    df_filtered3 = df_filtered2[df_filtered2.apply(lambda row: row['artist'].lower() in row['matched_artist'].lower(), axis=1)]
    
    output_path3 = path.with_name(path.stem.replace('_musicbrainz_enriched', '_filtered_country_remix_composer') + '.csv')
    df_filtered3.to_csv(output_path3, index=False, encoding='utf-8-sig')
    print(f'Saved {output_path3.name} ({len(df_filtered3)} rows)')
    print()

Processing sanjuanito_20260418_2309_musicbrainz_enriched.csv
Saved sanjuanito_20260418_2309_filtered_country.csv (113 rows)
Saved sanjuanito_20260418_2309_filtered_country_remix.csv (112 rows)
Saved sanjuanito_20260418_2309_filtered_country_remix_composer.csv (44 rows)

Processing tobas_20260418_2311_musicbrainz_enriched.csv
Saved tobas_20260418_2311_filtered_country.csv (114 rows)
Saved tobas_20260418_2311_filtered_country_remix.csv (110 rows)
Saved tobas_20260418_2311_filtered_country_remix_composer.csv (36 rows)

